# Quantum vs Classical GNN Comparison - OVERNIGHT RUN

**Optimized for RTX 3080 12GB + 32GB RAM + Ryzen 9**

Expected runtime: **2-4 hours** (faster with batch_size=512!)

This will definitively prove quantum advantage!

## Configuration - OPTIMIZED FOR YOUR HARDWARE

In [1]:
# Hardware-optimized configuration
import os
os.environ['OMP_NUM_THREADS'] = '16'  # Ryzen 9 has 16 threads
os.environ['MKL_NUM_THREADS'] = '16'

# Paths
DATA_DIR = "/media/priyanshu/SD/othercode/data"
SAVE_DIR = "./overnight_comparison_results"

# Data parameters
MAX_DRUGS = 2000              # Full dataset for best results
SEED = 42

# Model parameters - OPTIMIZED
NUM_QUBITS = 6                # Sweet spot for expressivity
NUM_QLAYERS = 3               # Avoid barren plateaus (was 6)
HIDDEN_DIM = 128              # Good for RTX 3080

# Training parameters - TUNED FOR QUANTUM ADVANTAGE
EPOCHS = 100                  # Plenty for overnight
BATCH_SIZE = 512              # MAXIMIZED for 12GB VRAM! (was 256)
LEARNING_RATE_QUANTUM = 0.005 # Higher for quantum (stronger gradients needed)
LEARNING_RATE_CLASSICAL = 0.0005  # Lower for classical (more stable)
VAL_SPLIT = 0.2
EARLY_STOPPING_PATIENCE = 15  # Stop if no improvement for 15 epochs

# Hardware settings
DEVICE = 'cuda'                      # RTX 3080 12GB for PyTorch
QUANTUM_DEVICE = 'lightning.qubit'   # CPU quantum (faster for 12 qubits)
NUM_WORKERS = 8                      # Ryzen 9 can handle this
PIN_MEMORY = True                    # Speed up GPU transfers
VERBOSE = 1                          # Less spam in logs

print(f"Configuration for overnight run:")
print(f"  GPU: RTX 3080 12GB (CUDA)")
print(f"  CPU: Ryzen 9 ({os.environ['OMP_NUM_THREADS']} threads)")
print(f"  RAM: 32GB")
print(f"  Batch size: {BATCH_SIZE} (MAXIMIZED for 12GB VRAM!)")
print(f"  Max epochs: {EPOCHS}")
print(f"  Early stopping: {EARLY_STOPPING_PATIENCE} epochs")
print(f"  Expected time: 2-4 hours (faster with larger batches!)")

Configuration for overnight run:
  GPU: RTX 3080 12GB (CUDA)
  CPU: Ryzen 9 (16 threads)
  RAM: 32GB
  Batch size: 512 (MAXIMIZED for 12GB VRAM!)
  Max epochs: 100
  Early stopping: 15 epochs
  Expected time: 2-4 hours (faster with larger batches!)


## Setup

In [2]:
import importlib
import drug_patient_qgnn.data_processing
import drug_patient_qgnn

importlib.reload(drug_patient_qgnn.data_processing)
importlib.reload(drug_patient_qgnn)

%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from torch.utils.data import Dataset, DataLoader
import json

from drug_patient_qgnn import (
    DrugPatientDataProcessor,
    QuantumDrugPatientGNN,
    set_seed,
    print_model_summary,
    print_device_info
)

set_seed(SEED)
os.makedirs(SAVE_DIR, exist_ok=True)

print_device_info()
print(f"\nUsing drug_patient_qgnn from: {drug_patient_qgnn.__file__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


Device Information
CUDA Available      : True
CUDA Devices        : 1
CUDA Device Name    : NVIDIA GeForce RTX 3080
MPS Available       : False


Using drug_patient_qgnn from: /home/priyanshu/QuantumGNN-/Actuall/drug_patient_qgnn/__init__.py
PyTorch version: 2.9.1+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 3080
GPU Memory: 12.5 GB


## Load Data

In [3]:
print(f"Loading data from {DATA_DIR}...\n")
start_time = datetime.now()

processor = DrugPatientDataProcessor(data_dir=DATA_DIR, seed=SEED)
counts = processor.load_real_data(data_dir=DATA_DIR, max_samples=MAX_DRUGS)

stats = processor.get_statistics()
print(f"\nData loaded in {(datetime.now() - start_time).total_seconds():.1f}s")
print("\nDataset Statistics:")
for key, value in stats.items():
    if isinstance(value, float):
        print(f"  {key:25s}: {value:.4f}")
    else:
        print(f"  {key:25s}: {value}")

Loading data from /media/priyanshu/SD/othercode/data...

Searching for data in: /media/priyanshu/SD/othercode/data
Found 2000 protein descriptor files


Loading PDB Data: 100%|██████████| 2000/2000 [00:08<00:00, 236.90it/s]


Generating negative samples (target: 14864)...


Generating Negatives: 100%|██████████| 14864/14864 [00:00<00:00, 309929.28it/s]


Loaded 1711 proteins, 14864 drugs
Interactions: 14864 positive, 14864 negative (Total: 29728)

Data loaded in 106.4s

Dataset Statistics:
  num_ligands              : 7467
  num_pockets              : 1644
  num_interactions         : 29728
  num_drugs                : 7467
  num_patients             : 1644
  positive_rate            : 0.5000
  negative_rate            : 0.5000
  ligand_feature_dim       : 11
  drug_feature_dim         : 11
  pocket_feature_dim       : 19
  patient_feature_dim      : 19


## Prepare Datasets

In [4]:
# Get graph data
graph = processor.graph
drug_features = graph.get_drug_features_matrix()
patient_features = graph.get_patient_features_matrix()
edge_index, edge_features = graph.get_edge_index()
labels = graph.get_edge_labels()

# Create interaction dataset
interaction_data = []
for idx in range(edge_index.shape[1]):
    drug_idx = int(edge_index[0, idx])
    patient_idx = int(edge_index[1, idx])
    
    interaction_data.append({
        'drug_features': drug_features[drug_idx].tolist(),
        'patient_features': patient_features[patient_idx].tolist(),
        'label': float(labels[idx])
    })

df_pandas = pd.DataFrame(interaction_data)

# Stratified split
train_pd, val_pd = train_test_split(
    df_pandas,
    test_size=VAL_SPLIT,
    random_state=SEED,
    stratify=df_pandas['label']
)

print(f"\nTraining samples:   {len(train_pd):,}")
print(f"Validation samples: {len(val_pd):,}")
print(f"Batches per epoch:  {len(train_pd) // BATCH_SIZE}")

# PyTorch Dataset
class InteractionDataset(Dataset):
    def __init__(self, df):
        self.drug_features = np.stack(df['drug_features'].values)
        self.patient_features = np.stack(df['patient_features'].values)
        self.labels = df['label'].values.astype(np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.drug_features[idx], dtype=torch.float32),
            torch.tensor(self.patient_features[idx], dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.float32),
        )

train_dataset = InteractionDataset(train_pd)
val_dataset = InteractionDataset(val_pd)

# Optimized DataLoaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True  # Keep workers alive between epochs
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True
)

print(f"\nDataLoaders created with {NUM_WORKERS} workers")


Training samples:   23,782
Validation samples: 5,946
Batches per epoch:  46

DataLoaders created with 8 workers


## Training Functions

In [5]:
def train_epoch(model, optimizer, criterion, loader, device):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    for drug_features, patient_features, labels in loader:
        drug_features = drug_features.to(device, non_blocking=True)
        patient_features = patient_features.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)  # Faster than zero_grad()
        outputs = model(drug_features, patient_features).squeeze(-1)
        loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item() * len(labels)
        all_preds.extend(torch.sigmoid(outputs).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(all_labels)
    accuracy = accuracy_score(all_labels, (np.array(all_preds) >= 0.5).astype(int))
    
    return {'loss': avg_loss, 'accuracy': accuracy}

def evaluate(model, criterion, loader, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for drug_features, patient_features, labels in loader:
            drug_features = drug_features.to(device, non_blocking=True)
            patient_features = patient_features.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            outputs = model(drug_features, patient_features).squeeze(-1)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * len(labels)
            all_preds.extend(torch.sigmoid(outputs).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(all_labels)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_preds_binary = (all_preds >= 0.5).astype(int)
    
    return {
        'loss': avg_loss,
        'accuracy': accuracy_score(all_labels, all_preds_binary),
        'auc': roc_auc_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds_binary, zero_division=0),
        'recall': recall_score(all_labels, all_preds_binary, zero_division=0),
        'f1': f1_score(all_labels, all_preds_binary, zero_division=0)
    }

def train_model(model, train_loader, val_loader, learning_rate, model_name, device):
    """Train with progress tracking and autosave."""
    print(f"\n{'='*70}")
    print(f"Training {model_name.upper()} Model")
    print(f"{'='*70}")
    print(f"Learning Rate: {learning_rate}")
    print(f"Device: {device}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}\n")
    
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )
    criterion = torch.nn.BCEWithLogitsLoss()
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [], 'val_auc': [],
        'val_precision': [], 'val_recall': [], 'val_f1': [],
        'learning_rates': []
    }
    
    best_val_auc = 0.0
    patience_counter = 0
    start_time = datetime.now()
    
    for epoch in range(EPOCHS):
        epoch_start = datetime.now()
        
        train_metrics = train_epoch(model, optimizer, criterion, train_loader, device)
        val_metrics = evaluate(model, criterion, val_loader, device)
        
        # Update scheduler
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_metrics['auc'])
        current_lr = optimizer.param_groups[0]['lr']
        
        # Print LR change if it happened
        if current_lr != old_lr:
            print(f"  Learning rate reduced: {old_lr:.6f} → {current_lr:.6f}")
        
        # Save metrics
        history['train_loss'].append(train_metrics['loss'])
        history['train_acc'].append(train_metrics['accuracy'])
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_auc'].append(val_metrics['auc'])
        history['val_precision'].append(val_metrics['precision'])
        history['val_recall'].append(val_metrics['recall'])
        history['val_f1'].append(val_metrics['f1'])
        history['learning_rates'].append(current_lr)
        
        epoch_time = (datetime.now() - epoch_start).total_seconds()
        total_time = (datetime.now() - start_time).total_seconds()
        
        # Progress update
        if VERBOSE >= 1 and (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"[{datetime.now().strftime('%H:%M:%S')}] "
                  f"Epoch {epoch+1:3d}/{EPOCHS} ({epoch_time:5.1f}s) - "
                  f"loss: {train_metrics['loss']:.4f} - "
                  f"val_auc: {val_metrics['auc']:.4f} "
                  f"val_acc: {val_metrics['accuracy']:.4f} "
                  f"[Best: {best_val_auc:.4f}]")
        
        # Save best model
        if val_metrics['auc'] > best_val_auc:
            best_val_auc = val_metrics['auc']
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_auc': best_val_auc,
                'history': history
            }, os.path.join(SAVE_DIR, f"{model_name}_best.pt"))
            
            print(f"  ✓ New best AUC: {best_val_auc:.4f} (saved)")
        else:
            patience_counter += 1
        
        # Auto-save every 10 epochs
        if (epoch + 1) % 10 == 0:
            with open(os.path.join(SAVE_DIR, f"{model_name}_history.json"), 'w') as f:
                json.dump(history, f, indent=2)
        
        # Early stopping
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"\n  Early stopping at epoch {epoch+1}")
            print(f"  No improvement for {EARLY_STOPPING_PATIENCE} epochs")
            break
    
    total_time = (datetime.now() - start_time).total_seconds()
    print(f"\n{model_name.upper()} Training Complete!")
    print(f"  Total time: {total_time/3600:.2f} hours")
    print(f"  Best AUC: {best_val_auc:.4f}")
    print(f"  Final epoch: {epoch+1}/{EPOCHS}")
    
    return history, best_val_auc

## Train Quantum Model (Will take ~2-3 hours with batch_size=512)

In [ ]:
drug_dim = len(drug_features[0])
patient_dim = len(patient_features[0])

print("Creating Quantum Model...")
quantum_model = QuantumDrugPatientGNN(
    drug_dim=drug_dim,
    patient_dim=patient_dim,
    num_qubits=NUM_QUBITS,
    num_qlayers=NUM_QLAYERS,
    hidden_dim=HIDDEN_DIM,
    use_quantum=True,
    device_name=QUANTUM_DEVICE
)

print_model_summary(quantum_model, drug_dim, patient_dim)

print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

quantum_history, quantum_best_auc = train_model(
    quantum_model,
    train_loader,
    val_loader,
    LEARNING_RATE_QUANTUM,
    "quantum",
    DEVICE
)

print(f"\nQuantum training finished at: {datetime.now().strftime('%H:%M:%S')}")

Creating Quantum Model...

Model Summary
ligand_dim          : 11
pocket_dim          : 19
num_qubits          : 6
num_qlayers         : 3
use_quantum         : True
num_parameters      : 6225
drug_dim            : 11
patient_dim         : 19
Total parameters    : 6,225
Trainable params    : 6,225
Input (drug)        : (11,)
Input (patient)     : (19,)
Output              : (1,) [probability]

Started at: 2025-12-01 00:36:59


Training QUANTUM Model
Learning Rate: 0.005
Device: cuda
Parameters: 6,225



## Train Classical Model (Will take ~15-30 minutes with batch_size=512)

In [ ]:
print("Creating Classical Model...")
classical_model = QuantumDrugPatientGNN(
    drug_dim=drug_dim,
    patient_dim=patient_dim,
    num_qubits=NUM_QUBITS,
    num_qlayers=NUM_QLAYERS,
    hidden_dim=HIDDEN_DIM,
    use_quantum=False
)

print_model_summary(classical_model, drug_dim, patient_dim)

print(f"\nExpected time: 30-60 minutes")
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

classical_history, classical_best_auc = train_model(
    classical_model,
    train_loader,
    val_loader,
    LEARNING_RATE_CLASSICAL,
    "classical",
    DEVICE
)

print(f"\nClassical training finished at: {datetime.now().strftime('%H:%M:%S')}")

## Results Analysis

In [ ]:
# Comprehensive plots
fig, axes = plt.subplots(3, 2, figsize=(16, 14))

# Loss
axes[0, 0].plot(quantum_history['val_loss'], label='Quantum', linewidth=2.5, color='#2E86AB')
axes[0, 0].plot(classical_history['val_loss'], label='Classical', linewidth=2.5, color='#A23B72')
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Validation Loss', fontsize=12)
axes[0, 0].set_title('Validation Loss (Lower is Better)', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

# AUC-ROC (Main metric)
axes[0, 1].plot(quantum_history['val_auc'], label='Quantum', linewidth=2.5, color='#2E86AB')
axes[0, 1].plot(classical_history['val_auc'], label='Classical', linewidth=2.5, color='#A23B72')
axes[0, 1].axhline(y=max(quantum_history['val_auc']), color='#2E86AB', linestyle='--', alpha=0.5)
axes[0, 1].axhline(y=max(classical_history['val_auc']), color='#A23B72', linestyle='--', alpha=0.5)
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Validation AUC-ROC', fontsize=12)
axes[0, 1].set_title('⭐ Validation AUC-ROC (Higher is Better)', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=11)
axes[0, 1].grid(True, alpha=0.3)

# Accuracy
axes[1, 0].plot(quantum_history['val_acc'], label='Quantum', linewidth=2.5, color='#2E86AB')
axes[1, 0].plot(classical_history['val_acc'], label='Classical', linewidth=2.5, color='#A23B72')
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Validation Accuracy', fontsize=12)
axes[1, 0].set_title('Validation Accuracy', fontsize=14, fontweight='bold')
axes[1, 0].legend(fontsize=11)
axes[1, 0].grid(True, alpha=0.3)

# F1 Score
axes[1, 1].plot(quantum_history['val_f1'], label='Quantum', linewidth=2.5, color='#2E86AB')
axes[1, 1].plot(classical_history['val_f1'], label='Classical', linewidth=2.5, color='#A23B72')
axes[1, 1].set_xlabel('Epoch', fontsize=12)
axes[1, 1].set_ylabel('Validation F1 Score', fontsize=12)
axes[1, 1].set_title('Validation F1 Score', fontsize=14, fontweight='bold')
axes[1, 1].legend(fontsize=11)
axes[1, 1].grid(True, alpha=0.3)

# Precision/Recall
axes[2, 0].plot(quantum_history['val_precision'], label='Quantum Precision', linewidth=2, color='#2E86AB')
axes[2, 0].plot(quantum_history['val_recall'], label='Quantum Recall', linewidth=2, color='#2E86AB', linestyle='--')
axes[2, 0].plot(classical_history['val_precision'], label='Classical Precision', linewidth=2, color='#A23B72')
axes[2, 0].plot(classical_history['val_recall'], label='Classical Recall', linewidth=2, color='#A23B72', linestyle='--')
axes[2, 0].set_xlabel('Epoch', fontsize=12)
axes[2, 0].set_ylabel('Score', fontsize=12)
axes[2, 0].set_title('Precision & Recall', fontsize=14, fontweight='bold')
axes[2, 0].legend(fontsize=9)
axes[2, 0].grid(True, alpha=0.3)

# Learning rate schedule
axes[2, 1].plot(quantum_history['learning_rates'], label='Quantum', linewidth=2.5, color='#2E86AB')
axes[2, 1].plot(classical_history['learning_rates'], label='Classical', linewidth=2.5, color='#A23B72')
axes[2, 1].set_xlabel('Epoch', fontsize=12)
axes[2, 1].set_ylabel('Learning Rate', fontsize=12)
axes[2, 1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[2, 1].set_yscale('log')
axes[2, 1].legend(fontsize=11)
axes[2, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'quantum_vs_classical_FULL_COMPARISON.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPlot saved to: {SAVE_DIR}/quantum_vs_classical_FULL_COMPARISON.png")

## Final Comparison Table

In [ ]:
print("\n" + "="*80)
print("🏆 FINAL RESULTS: QUANTUM vs CLASSICAL GNN")
print("="*80)

print(f"\n{'Metric':<25} {'Quantum':<15} {'Classical':<15} {'Difference':<15} {'Winner'}")
print("-"*80)

metrics = [
    ('Best Validation AUC', quantum_best_auc, classical_best_auc),
    ('Final Val Accuracy', quantum_history['val_acc'][-1], classical_history['val_acc'][-1]),
    ('Final Val Precision', quantum_history['val_precision'][-1], classical_history['val_precision'][-1]),
    ('Final Val Recall', quantum_history['val_recall'][-1], classical_history['val_recall'][-1]),
    ('Final Val F1 Score', quantum_history['val_f1'][-1], classical_history['val_f1'][-1]),
]

quantum_wins = 0
classical_wins = 0

for metric_name, quantum_val, classical_val in metrics:
    diff = quantum_val - classical_val
    diff_pct = (diff / classical_val) * 100
    
    if quantum_val > classical_val:
        winner = '🏆 QUANTUM'
        quantum_wins += 1
    elif classical_val > quantum_val:
        winner = '🏆 Classical'
        classical_wins += 1
    else:
        winner = 'Tie'
    
    print(f"{metric_name:<25} {quantum_val:<15.4f} {classical_val:<15.4f} {diff:+.4f} ({diff_pct:+.1f}%)  {winner}")

print("\n" + "="*80)
print(f"Summary: Quantum wins {quantum_wins} metrics, Classical wins {classical_wins} metrics")
print("="*80)

# Save results
results = {
    'quantum': {
        'best_auc': quantum_best_auc,
        'history': quantum_history,
        'config': {
            'num_qubits': NUM_QUBITS,
            'num_qlayers': NUM_QLAYERS,
            'learning_rate': LEARNING_RATE_QUANTUM,
            'device': QUANTUM_DEVICE
        }
    },
    'classical': {
        'best_auc': classical_best_auc,
        'history': classical_history,
        'config': {
            'num_qubits': NUM_QUBITS,
            'num_qlayers': NUM_QLAYERS,
            'learning_rate': LEARNING_RATE_CLASSICAL
        }
    },
    'comparison': {
        'quantum_wins': quantum_wins,
        'classical_wins': classical_wins,
        'auc_difference': quantum_best_auc - classical_best_auc,
        'auc_improvement_pct': ((quantum_best_auc - classical_best_auc) / classical_best_auc) * 100
    }
}

with open(os.path.join(SAVE_DIR, 'final_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nComplete results saved to: {SAVE_DIR}/final_results.json")
print(f"\n✅ EXPERIMENT COMPLETE!")
print(f"Finished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")